In [ ]:
# ---- Clean + Stable Environment Setup ----
!pip -q uninstall -y sentence-transformers
!pip -q install "jedi>=0.16"

# Upgrade build tools (prevents wheel build issues)
!pip -q install --upgrade pip setuptools wheel

# Ensure compatible pandas version for Colab
!pip -q install pandas==2.2.2

# Install HuggingFace + evaluation stack with safe pinned versions
!pip -q install \
    tokenizers==0.15.2 \
    transformers==4.39.3 \
    accelerate \
    datasets \
    bert-score \
    evaluate \
    sentencepiece \
    tqdm

# ---- Verify versions ----
import torch, pandas, transformers, tokenizers
print("torch:", torch.__version__)
print("pandas:", pandas.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

unseen_dir= "/content/drive/MyDrive/biomedical_text_generation/data/unseen/title_text_gen_unseen.jsonl"

# Folder containing: state.json, dataset_info.json, and the arrow shards
TEST_DATASET_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/biov2bart_text_gen/test"
#TEST_DATASET_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/biot5_text_gen/test"

# Folder containing your fine-tuned checkpoint (config.json, model weights, etc.)
CHECKPOINT_DIR = "/content/drive/MyDrive/biomedical_text_generation/models/biov2bart_title_text_gen_final"
#CHECKPOINT_DIR = "/content/drive/MyDrive/biomedical_text_generation/models/biot5_title_text_gen_final"

OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biov2bart"
#OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biot5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generation / eval params
MAX_INPUT_LEN  = 128
MAX_NEW_TOKENS = 1024
BATCH_SIZE     = 8

# BERTScore model (biomedical-friendly)
BERTSCORE_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
LANG = "en"


In [ ]:
############## testing ############
import json
from datasets import load_from_disk
ds = load_from_disk(TEST_DATASET_DIR)
print("Loaded:", ds)

#
jsonl_path = unseen_dir

with open(jsonl_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total rows:", len(lines))

# Load one example (e.g., index 100)
ex = json.loads(lines[100])

print("Columns:", ex.keys())
print("\nExample input:", str(ex.get("input"))[:500])
print("\nExample target:", str(ex.get("target"))[:1500])

In [ ]:
import json
import numpy as np

jsonl_path = unseen_dir


input_lengths = []
target_lengths = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        example = json.loads(line)

        # Safely get fields (in case some rows are missing them)
        inp = example.get("input", "")
        tgt = example.get("target", "")

        input_lengths.append(len(inp.split()))
        target_lengths.append(len(tgt.split()))

print("Total examples:", len(input_lengths))
print("Mean input length:", np.mean(input_lengths))
print("Mean target length:", np.mean(target_lengths))


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

jsonl_path = unseen_dir

input_lengths = []
target_lengths = []

# Read JSONL file
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        example = json.loads(line)

        inp = example.get("input", "")
        tgt = example.get("target", "")

        input_lengths.append(len(inp.split()))
        target_lengths.append(len(tgt.split()))

print("Total examples:", len(input_lengths))

# ---- Input Histogram ----
plt.figure()
plt.hist(input_lengths, bins=50)
plt.title("Distribution of Input Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Target Histogram ----
plt.figure()
plt.hist(target_lengths, bins=50)
plt.title("Distribution of Target Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Descriptive Stats ----
print("Input Mean:", np.mean(input_lengths))
print("Input Median:", np.median(input_lengths))

print("Target Mean:", np.mean(target_lengths))
print("Target Median:", np.median(target_lengths))

In [ ]:
# ---- Input Boxplot ----
plt.figure()
plt.boxplot(input_lengths)
plt.title("Boxplot of Input Word Counts")
plt.ylabel("Number of Words")
plt.show()

# ---- Target Boxplot ----
plt.figure()
plt.boxplot(target_lengths)
plt.title("Boxplot of Target Word Counts")
plt.ylabel("Number of Words")
plt.show()


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR, use_fast=False) # true for Biov2Bart
model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT_DIR).to(device)
model.eval()


In [ ]:
from tqdm.auto import tqdm
NUM_BEAMS = 4

preds = []
refs = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        example = json.loads(line)
        refs.append(str(example.get("target", "")))

print("Total references:", len(refs))
print("Sample ref:", refs[0][:500])

@torch.no_grad()
def generate_from_tokens(batch):
    input_ids = torch.tensor(batch["input_ids"]).to(device)
    attn_mask = torch.tensor(batch["attention_mask"]).to(device)

    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attn_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        early_stopping=True
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

for i in tqdm(range(0, len(ds), BATCH_SIZE), desc="Generating"):
    batch = ds[i:i+BATCH_SIZE]
    preds.extend(generate_from_tokens(batch))

print("Generated:", len(preds))
print("\nREF:", refs[0][:300])
print("\nPRED:", preds[0][:300])


In [ ]:
print("Generated:", len(preds))
print("\nREF:", refs[1020][:3000])
print("\nPRED:", preds[1020][:3000])


In [ ]:
from transformers import AutoTokenizer
from bert_score import score

BERTSCORE_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
NUM_LAYERS = 12

bs_tokenizer = AutoTokenizer.from_pretrained(BERTSCORE_MODEL, use_fast=True)

def truncate_for_bertscore(texts, tokenizer, max_len=512):
    out = []
    for t in texts:
        ids = tokenizer.encode(t, add_special_tokens=True, truncation=True, max_length=max_len)
        out.append(tokenizer.decode(ids, skip_special_tokens=True))
    return out

# Truncate both predictions and references to BERT's max length
preds_trunc = truncate_for_bertscore(preds, bs_tokenizer, max_len=512)
refs_trunc  = truncate_for_bertscore(refs,  bs_tokenizer, max_len=512)

P, R, F1 = score(
    cands=preds_trunc,
    refs=refs_trunc,
    model_type=BERTSCORE_MODEL,
    num_layers=NUM_LAYERS,
    lang="en",
    verbose=True,
    rescale_with_baseline=False,
    use_fast_tokenizer=False,
    batch_size=16  # lower if OOM
)

print("BERTScore means (truncated to 512 WP tokens):")
print("  Precision:", float(P.mean()))
print("  Recall:   ", float(R.mean()))
print("  F1:       ", float(F1.mean()))


In [ ]:
import pandas as pd, json

import os

jsonl_path = unseen_dir

inputs = []
refs = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        example = json.loads(line)
        inputs.append(str(example.get("input", "")))
        refs.append(str(example.get("target", "")))


out_df = pd.DataFrame({
    "input": inputs,
    "target": refs,
    "prediction": preds,
    "bertscore_P": P.cpu().numpy(),
    "bertscore_R": R.cpu().numpy(),
    "bertscore_F1": F1.cpu().numpy(),
})

csv_path = os.path.join(OUTPUT_DIR, "test_with_bertscore.csv")
out_df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

summary = {
    "checkpoint_dir": CHECKPOINT_DIR,
    "test_dataset_dir": TEST_DATASET_DIR,
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS
    },
    "bertscore_model": BERTSCORE_MODEL,
    "mean_precision": float(P.mean()),
    "mean_recall": float(R.mean()),
    "mean_f1": float(F1.mean()),
    "n_examples": int(len(out_df)),
}

json_path = os.path.join(OUTPUT_DIR, "bertscore_title_text_gen.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:", json_path)
out_df.head()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set this exactly as you saved it
OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biot5"
csv_path = os.path.join(OUTPUT_DIR, "test_with_bertscore.csv")

df = pd.read_csv(csv_path)

print("Loaded rows:", len(df))

# ---- Mean BERTScores ----
mean_P = float(df["bertscore_P"].mean())
mean_R = float(df["bertscore_R"].mean())
mean_F1 = float(df["bertscore_F1"].mean())

print("\n--- Mean BERTScores ---")
print("Mean Precision:", mean_P)
print("Mean Recall:   ", mean_R)
print("Mean F1:       ", mean_F1)

# ---- Word Counts ----
input_wc = [len(str(x).split()) for x in df["input"]]
target_wc = [len(str(x).split()) for x in df["target"]]
pred_wc = [len(str(x).split()) for x in df["prediction"]]

# ---- Histogram: Input ----
plt.figure()
plt.hist(input_wc, bins=50)
plt.title("Distribution of Input Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Histogram: Target ----
plt.figure()
plt.hist(target_wc, bins=50)
plt.title("Distribution of Target Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Histogram: Generated ----
plt.figure()
plt.hist(pred_wc, bins=50)
plt.title("Distribution of Generated Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Word Count Statistics ----
print("\n--- Word Count Statistics (mean / median) ---")
print("Input     :", float(np.mean(input_wc)), "/", float(np.median(input_wc)))
print("Target    :", float(np.mean(target_wc)), "/", float(np.median(target_wc)))
print("Generated :", float(np.mean(pred_wc)), "/", float(np.median(pred_wc)))


In [ ]:
# ---- Compute Word Counts ----
input_wc = np.array([len(str(x).split()) for x in df["input"]])
target_wc = np.array([len(str(x).split()) for x in df["target"]])
pred_wc = np.array([len(str(x).split()) for x in df["prediction"]])

# ---- Differences ----
diff_input_target = input_wc - target_wc
diff_target_generated = target_wc - pred_wc

# ---- Histogram: Input - Target ----
plt.figure()
plt.hist(diff_input_target, bins=50)
plt.title("Word Count Difference: Input - Target")
plt.xlabel("Word Difference (negative = target longer)")
plt.ylabel("Frequency")
plt.show()

# ---- Histogram: Target - Generated ----
plt.figure()
plt.hist(diff_target_generated, bins=50)
plt.title("Word Count Difference: Target - Generated")
plt.xlabel("Word Difference (negative = generated longer)")
plt.ylabel("Frequency")
plt.show()

# ---- Summary Statistics ----
print("\n--- Difference Statistics ---")
print("Input - Target  | Mean:", float(np.mean(diff_input_target)),
      "| Median:", float(np.median(diff_input_target)))

print("Target - Gen    | Mean:", float(np.mean(diff_target_generated)),
      "| Median:", float(np.median(diff_target_generated)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Word counts ----
input_wc = np.array([len(str(x).split()) for x in df["input"]])
pred_wc = np.array([len(str(x).split()) for x in df["prediction"]])

# ---- Difference ----
diff_input_generated = input_wc - pred_wc

# ---- Histogram ----
plt.figure()
plt.hist(diff_input_generated, bins=50)

plt.title("Word Count Difference: Input - Generated")
plt.xlabel("Word Difference (negative = generated longer)")
plt.ylabel("Frequency")

plt.show()

# ---- Summary statistics ----
print("\n--- Input vs Generated Statistics ---")

print("Mean compression:", float(np.mean(diff_input_generated)))
print("Median compression:", float(np.median(diff_input_generated)))

print("Average input length:", float(np.mean(input_wc)))
print("Average generated length:", float(np.mean(pred_wc)))

compression_ratio = np.mean(pred_wc) / np.mean(input_wc)

print("Compression ratio (generated/input):", compression_ratio)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set this exactly as you saved it
OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biov2bart"
csv_path = os.path.join(OUTPUT_DIR, "test_with_bertscore.csv")

df = pd.read_csv(csv_path)

print("Loaded rows:", len(df))

# ---- Mean BERTScores ----
mean_P = float(df["bertscore_P"].mean())
mean_R = float(df["bertscore_R"].mean())
mean_F1 = float(df["bertscore_F1"].mean())

print("\n--- Mean BERTScores ---")
print("Mean Precision:", mean_P)
print("Mean Recall:   ", mean_R)
print("Mean F1:       ", mean_F1)

# ---- Word Counts ----
input_wc = [len(str(x).split()) for x in df["input"]]
target_wc = [len(str(x).split()) for x in df["target"]]
pred_wc = [len(str(x).split()) for x in df["prediction"]]

# ---- Histogram: Input ----
plt.figure()
plt.hist(input_wc, bins=50)
plt.title("Distribution of Input Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Histogram: Target ----
plt.figure()
plt.hist(target_wc, bins=50)
plt.title("Distribution of Target Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Histogram: Generated ----
plt.figure()
plt.hist(pred_wc, bins=50)
plt.title("Distribution of Generated Word Counts")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.show()

# ---- Word Count Statistics ----
print("\n--- Word Count Statistics (mean / median) ---")
print("Input     :", float(np.mean(input_wc)), "/", float(np.median(input_wc)))
print("Target    :", float(np.mean(target_wc)), "/", float(np.median(target_wc)))
print("Generated :", float(np.mean(pred_wc)), "/", float(np.median(pred_wc)))


In [ ]:
# ---- Compute Word Counts ----
input_wc = np.array([len(str(x).split()) for x in df["input"]])
target_wc = np.array([len(str(x).split()) for x in df["target"]])
pred_wc = np.array([len(str(x).split()) for x in df["prediction"]])

# ---- Differences ----
diff_input_target = input_wc - target_wc
diff_target_generated = target_wc - pred_wc

# ---- Histogram: Input - Target ----
plt.figure()
plt.hist(diff_input_target, bins=50)
plt.title("Word Count Difference: Input - Target")
plt.xlabel("Word Difference (negative = target longer)")
plt.ylabel("Frequency")
plt.show()

# ---- Histogram: Target - Generated ----
plt.figure()
plt.hist(diff_target_generated, bins=50)
plt.title("Word Count Difference: Target - Generated")
plt.xlabel("Word Difference (negative = generated longer)")
plt.ylabel("Frequency")
plt.show()

# ---- Summary Statistics ----
print("\n--- Difference Statistics ---")
print("Input - Target  | Mean:", float(np.mean(diff_input_target)),
      "| Median:", float(np.median(diff_input_target)))

print("Target - Gen    | Mean:", float(np.mean(diff_target_generated)),
      "| Median:", float(np.median(diff_target_generated)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Word counts ----
input_wc = np.array([len(str(x).split()) for x in df["input"]])
pred_wc = np.array([len(str(x).split()) for x in df["prediction"]])

# ---- Difference ----
diff_input_generated = input_wc - pred_wc

# ---- Histogram ----
plt.figure()
plt.hist(diff_input_generated, bins=50)

plt.title("Word Count Difference: Input - Generated")
plt.xlabel("Word Difference (negative = generated longer)")
plt.ylabel("Frequency")

plt.show()

# ---- Summary statistics ----
print("\n--- Input vs Generated Statistics ---")

print("Mean compression:", float(np.mean(diff_input_generated)))
print("Median compression:", float(np.median(diff_input_generated)))

print("Average input length:", float(np.mean(input_wc)))
print("Average generated length:", float(np.mean(pred_wc)))

compression_ratio = np.mean(pred_wc) / np.mean(input_wc)

print("Compression ratio (generated/input):", compression_ratio)

In [ ]:
# Robust paired evaluation for two CSVs (Colab-ready)
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon, binomtest

# paths (keep yours)
BIOBART_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biov2bart/test_with_bertscore.csv"
BIOT5_PATH   = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biot5/test_with_bertscore.csv"

# Load CSVs
df_a = pd.read_csv(BIOBART_PATH)  # BioBART
df_b = pd.read_csv(BIOT5_PATH)    # BioT5

print("Rows in BioBART file:", len(df_a))
print("Rows in BioT5 file:  ", len(df_b))

# ---- Robust alignment (handles duplicates) ----
# Add occurrence index within each (input, target) group
df_a = df_a.copy()
df_b = df_b.copy()

df_a["_k"] = df_a.groupby(["input", "target"]).cumcount()
df_b["_k"] = df_b.groupby(["input", "target"]).cumcount()

# 1-to-1 merge on (input, target, _k)
df = df_a.merge(
    df_b,
    on=["input", "target", "_k"],
    suffixes=("_biobart", "_biot5"),
    how="inner"
)

print("Number of aligned samples after occ-index merge:", len(df))

# (optional) cleanup: drop the helper index if you like
df = df.drop(columns=["_k"])

# ---- Metrics to evaluate ----
metrics = ["bertscore_P", "bertscore_R", "bertscore_F1"]

# helper functions
def cohens_d_paired(diff_arr: np.ndarray) -> float:
    # use sample std (ddof=1)
    sd = np.std(diff_arr, ddof=1)
    return np.nan if sd == 0 else diff_arr.mean() / sd

def bootstrap_mean_diff_ci(diff_arr: np.ndarray, n_boot=10000, seed=123):
    rng = np.random.default_rng(seed)
    n = len(diff_arr)
    # vectorized bootstrap: draw indices and compute means
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diff_arr[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return boot_means.mean(), lo, hi

# run stats for each metric and print nicely
results = []
for metric in metrics:
    col_a = f"{metric}_biobart"
    col_b = f"{metric}_biot5"

    if col_a not in df.columns or col_b not in df.columns:
        print(f"Skipping {metric}: columns {col_a} or {col_b} not found.")
        continue

    scores_a = df[col_a].to_numpy(dtype=float)
    scores_b = df[col_b].to_numpy(dtype=float)
    diff = scores_a - scores_b

    # basics
    mean_a, std_a = scores_a.mean(), scores_a.std(ddof=1)
    mean_b, std_b = scores_b.mean(), scores_b.std(ddof=1)
    mean_diff = float(diff.mean())
    median_diff = float(np.median(diff))

    # tests
    t_stat, p_t = ttest_rel(scores_a, scores_b, nan_policy="omit")

    try:
        w_stat, p_w = wilcoxon(scores_a, scores_b, zero_method="wilcox")
    except ValueError:
        # e.g., all diffs zero -> wilcoxon can't run
        w_stat, p_w = np.nan, np.nan

    wins = int((diff > 0).sum())
    losses = int((diff < 0).sum())
    ties = int((diff == 0).sum())
    total_non_ties = wins + losses
    win_rate = np.nan if total_non_ties == 0 else wins / total_non_ties
    sign_p = float(binomtest(wins, total_non_ties, 0.5).pvalue) if total_non_ties > 0 else np.nan

    d = cohens_d_paired(diff)

    # bootstrap CI
    boot_mean, ci_lo, ci_hi = bootstrap_mean_diff_ci(diff, n_boot=10000, seed=123)

    res = {
        "metric": metric,
        "BioBART_mean": mean_a,
        "BioBART_std": std_a,
        "BioT5_mean": mean_b,
        "BioT5_std": std_b,
        "mean_diff": mean_diff,
        "median_diff": median_diff,
        "t_stat": float(t_stat),
        "t_p": float(p_t),
        "wilcoxon_p": float(p_w) if not np.isnan(p_w) else np.nan,
        "bootstrap_mean_diff": float(boot_mean),
        "bootstrap_CI_low": float(ci_lo),
        "bootstrap_CI_high": float(ci_hi),
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_non_ties": float(win_rate) if not np.isnan(win_rate) else np.nan,
        "sign_test_p": float(sign_p) if not np.isnan(sign_p) else np.nan,
        "cohens_d": float(d) if not np.isnan(d) else np.nan,
        "n": len(diff)
    }

    results.append(res)

    # print a paper-friendly block for this metric
    print("\n" + "="*60)
    print(f"Metric: {metric}")
    print(f"Samples (N): {len(diff)}")
    print(f"BioBART: {mean_a:.4f} ± {std_a:.4f}")
    print(f"BioT5:   {mean_b:.4f} ± {std_b:.4f}")
    print(f"Δ (BioBART − BioT5): mean={mean_diff:.4f}, median={median_diff:.4f}")
    print(f"Paired t-test: t={t_stat:.4f}, p={p_t:.4g}")
    print(f"Wilcoxon (paired): p={p_w:.4g}")
    print(f"Bootstrap mean Δ: {boot_mean:.6f}, 95% CI = [{ci_lo:.6f}, {ci_hi:.6f}]")
    print(f"Wins/losses/ties: {wins}/{losses}/{ties}  (win_rate_excluding_ties={win_rate:.3f})")
    print(f"Sign test (binomial) p = {sign_p:.4g}")
    print(f"Cohen's d (paired) = {d:.3f}")
    print("="*60)

# optional: put full results into a DataFrame for easy saving
results_df = pd.DataFrame(results)
results_df

In [ ]:
mean_biobart = df["bertscore_F1_biobart"].mean()
mean_biot5 = df["bertscore_F1_biot5"].mean()

improvement = (mean_biobart - mean_biot5) / mean_biot5 * 100

print(f"Relative improvement (F1): {improvement:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon, binomtest

# Paths to your CSVs
BIOBART_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biov2bart/test_with_bertscore.csv"
BIOT5_PATH   = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biot5/test_with_bertscore.csv"

df_a = pd.read_csv(BIOBART_PATH)
df_b = pd.read_csv(BIOT5_PATH)

# ---- FIX: robust 1-to-1 alignment even with duplicates ----
df_a = df_a.copy()
df_b = df_b.copy()

df_a["_k"] = df_a.groupby(["input", "target"]).cumcount()
df_b["_k"] = df_b.groupby(["input", "target"]).cumcount()

df = df_a.merge(
    df_b,
    on=["input", "target", "_k"],
    suffixes=("_biobart", "_biot5"),
    how="inner"
).drop(columns=["_k"])

print("Number of aligned samples:", len(df))  # should be 3777

metrics = ["bertscore_P", "bertscore_R", "bertscore_F1"]
results = []

# Bootstrap settings
n_boot = 10000
rng = np.random.default_rng(123)

for metric in metrics:
    scores_a = df[f"{metric}_biobart"].to_numpy(dtype=float)
    scores_b = df[f"{metric}_biot5"].to_numpy(dtype=float)
    diff = scores_a - scores_b

    # Mean/std (sample std, ddof=1)
    mean_a = float(scores_a.mean())
    std_a  = float(scores_a.std(ddof=1))

    mean_b = float(scores_b.mean())
    std_b  = float(scores_b.std(ddof=1))

    # Paired t-test
    t_stat, p_ttest = ttest_rel(scores_a, scores_b)

    # Wilcoxon signed-rank (paired, nonparametric)
    w_stat, p_wilcoxon = wilcoxon(scores_a, scores_b, zero_method="wilcox")

    # Win rate + sign test
    wins = int((diff > 0).sum())
    losses = int((diff < 0).sum())
    ties = int((diff == 0).sum())

    non_ties = wins + losses
    win_rate = np.nan if non_ties == 0 else wins / non_ties
    sign_p = np.nan if non_ties == 0 else float(binomtest(wins, non_ties, 0.5).pvalue)

    # Cohen's d (paired): mean(diff) / std(diff)
    sd_diff = float(diff.std(ddof=1))
    cohens_d = np.nan if sd_diff == 0 else float(diff.mean() / sd_diff)

    # Bootstrap 95% CI of mean(diff) (vectorized)
    n = len(diff)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diff[idx].mean(axis=1)
    lower, upper = np.percentile(boot_means, [2.5, 97.5])

    results.append({
        "metric": metric,
        "BioBART_mean": mean_a,
        "BioBART_std": std_a,
        "BioT5_mean": mean_b,
        "BioT5_std": std_b,
        "mean_diff": float(diff.mean()),
        "t-test_p": float(p_ttest),
        "wilcoxon_p": float(p_wilcoxon),
        "bootstrap_CI_low": float(lower),
        "bootstrap_CI_high": float(upper),
        "win_rate_excl_ties": float(win_rate) if not np.isnan(win_rate) else np.nan,
        "sign_test_p": float(sign_p) if not np.isnan(sign_p) else np.nan,
        "cohens_d": float(cohens_d) if not np.isnan(cohens_d) else np.nan,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "n": int(n)
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
for _, r in results_df.iterrows():

    print(f"\nMetric: {r['metric']}")

    print(f"BioBART: {r['BioBART_mean']:.4f} ± {r['BioBART_std']:.4f}")
    print(f"BioT5:   {r['BioT5_mean']:.4f} ± {r['BioT5_std']:.4f}")

    print(f"Δ (BioBART - BioT5): {r['mean_diff']:.4f}")

    print(f"t-test p = {r['t-test_p']:.4g}")
    print(f"Wilcoxon p = {r['wilcoxon_p']:.4g}")

    print(f"Bootstrap 95% CI: [{r['bootstrap_CI_low']:.4f}, {r['bootstrap_CI_high']:.4f}]")

    print(f"Win rate (excluding ties): {r['win_rate_excl_ties']:.3f}")
    print(f"Sign test p = {r['sign_test_p']:.4g}")

    print(f"Cohen's d = {r['cohens_d']:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import probplot

metric = "bertscore_F1"  # or bertscore_P / bertscore_R

a = df[f"{metric}_biobart"].to_numpy(dtype=float)
b = df[f"{metric}_biot5"].to_numpy(dtype=float)
diff = a - b

wins = (diff > 0).sum()
losses = (diff < 0).sum()
ties = (diff == 0).sum()
non_ties = wins + losses
win_rate = wins / non_ties if non_ties > 0 else np.nan

print("N:", len(diff))
print("Mean diff (A-B):", diff.mean())
print("Median diff (A-B):", np.median(diff))
print(f"Wins/Losses/Ties: {wins}/{losses}/{ties}")
print("Win rate (A>B, excluding ties):", win_rate)

In [ ]:
plt.figure()
plt.hist(diff, bins=40)
plt.axvline(0, linestyle="--")
plt.title(f"Histogram of paired differences (Δ = BioBART − BioT5) — {metric}")
plt.xlabel("Δ")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure()
plt.scatter(b, a, s=10, alpha=0.6)
mn = min(a.min(), b.min())
mx = max(a.max(), b.max())
plt.plot([mn, mx], [mn, mx], linestyle="--")  # diagonal y=x
plt.title(f"Per-example scores — {metric}\nPoints above diagonal = BioBART better")
plt.xlabel("BioT5 score")
plt.ylabel("BioBART score")
plt.show()

In [ ]:
plt.figure()
plt.boxplot(diff, vert=True, showmeans=True)
plt.axhline(0, linestyle="--")
plt.title(f"Boxplot of Δ (BioBART − BioT5) — {metric}")
plt.ylabel("Δ")
plt.xticks([1], [metric])
plt.show()

In [ ]:
n_boot = 10000
rng = np.random.default_rng(123)

idx = rng.integers(0, len(diff), size=(n_boot, len(diff)))
boot_means = diff[idx].mean(axis=1)

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

plt.figure()
plt.hist(boot_means, bins=50)
plt.axvline(0, linestyle="--")
plt.axvline(ci_low, linestyle="--")
plt.axvline(ci_high, linestyle="--")
plt.title(f"Bootstrap of mean Δ — {metric}\n95% CI [{ci_low:.4f}, {ci_high:.4f}]")
plt.xlabel("Bootstrap mean Δ")
plt.ylabel("Count")
plt.show()

print("Bootstrap mean Δ:", boot_means.mean())
print("95% CI:", (ci_low, ci_high))

In [ ]:
plt.figure()
probplot(diff, dist="norm", plot=plt)
plt.title(f"Q–Q plot of Δ (normality check) — {metric}")
plt.show()

In [ ]:
!pip -q install "jedi>=0.16"
!pip -q install -U pip setuptools wheel
!pip -q install "numpy==1.26.4" "pandas>=2.0"

# spaCy + scispaCy stack (no scipy)
!pip -q install "spacy==3.7.5" "thinc==8.2.5" "scispacy==0.5.4"

# SciSpaCy model
!pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_scibert-0.5.4.tar.gz

In [ ]:
!pip install spacy
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_lg-0.5.1.tar.gz

In [ ]:
import numpy as np
import pandas as pd
import spacy
import scispacy
from scispacy.linking import EntityLinker

BIOBART_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biov2bart/test_with_bertscore.csv"

#BIOT5_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biot5/test_with_bertscore.csv"

df = pd.read_csv(BIOBART_PATH)

# -----------------------------
# Load model + add UMLS linker (NEW spaCy API)
# -----------------------------
nlp = spacy.load("en_core_sci_scibert")

# Add the scispaCy UMLS linker via factory name
# max_entities_per_mention=1 keeps top candidate only
nlp.add_pipe(
    "scispacy_linker",
    config={
        "linker_name": "umls",
        "resolve_abbreviations": True,
        "max_entities_per_mention": 1,
    },
)

print("Pipes:", nlp.pipe_names)

# -----------------------------
# Helpers
# -----------------------------
def cuis_from_doc(doc) -> set:
    """Extract unique UMLS CUIs from a processed spaCy doc."""
    cuis = set()
    for ent in doc.ents:
        # scispacy_linker stores candidates here
        if hasattr(ent._, "kb_ents"):
            for cui, score in ent._.kb_ents:
                cuis.add(cui)
    return cuis

def prf1(ref_set: set, pred_set: set):
    """
    Concept-level PRF1 based on set overlap.
    ref_set = CUIs from the "reference" text
    pred_set = CUIs from the "predicted/compared" text
    """
    if len(ref_set) == 0 and len(pred_set) == 0:
        return 1.0, 1.0, 1.0
    if len(ref_set) == 0 or len(pred_set) == 0:
        return 0.0, 0.0, 0.0

    tp = len(ref_set & pred_set)
    p = tp / len(pred_set) if len(pred_set) else 0.0
    r = tp / len(ref_set) if len(ref_set) else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
    return p, r, f1

def extract_cui_sets(texts, batch_size=32):
    texts = pd.Series(texts).fillna("").astype(str).tolist()
    return [cuis_from_doc(doc) for doc in nlp.pipe(texts, batch_size=batch_size)]

def compare_concepts(df_in, col_ref, col_pred, name, batch_size=32):
    """
    Compare CUIs between df_in[col_ref] (reference) and df_in[col_pred] (compared).
    Returns per-example CUIs + PRF1 and macro/micro aggregates.
    """
    ref_sets = extract_cui_sets(df_in[col_ref], batch_size=batch_size)
    pred_sets = extract_cui_sets(df_in[col_pred], batch_size=batch_size)

    P, R, F1 = [], [], []
    TP = FP = FN = 0

    for rs, ps in zip(ref_sets, pred_sets):
        p, r, f = prf1(rs, ps)
        P.append(p); R.append(r); F1.append(f)

        TP += len(rs & ps)
        FP += len(ps - rs)
        FN += len(rs - ps)

    out = pd.DataFrame({
        f"{name}_ref_cuis":  [";".join(sorted(s)) for s in ref_sets],
        f"{name}_pred_cuis": [";".join(sorted(s)) for s in pred_sets],
        f"{name}_P": P,
        f"{name}_R": R,
        f"{name}_F1": F1,
    })

    macro_P = float(np.mean(P))
    macro_R = float(np.mean(R))
    macro_F1 = float(np.mean(F1))

    micro_P = TP / (TP + FP) if (TP + FP) else 0.0
    micro_R = TP / (TP + FN) if (TP + FN) else 0.0
    micro_F1 = (2 * micro_P * micro_R / (micro_P + micro_R)) if (micro_P + micro_R) else 0.0

    summary = {
        "comparison": name,
        "N": int(len(df_in)),
        "macro_P": macro_P,
        "macro_R": macro_R,
        "macro_F1": macro_F1,
        "micro_P": float(micro_P),
        "micro_R": float(micro_R),
        "micro_F1": float(micro_F1),
    }
    return out, summary

# -----------------------------
# RUN: 3 comparisons you asked for
# -----------------------------
#out_it, sum_it = compare_concepts(df, "input", "target", "input_vs_target")
out_tg, sum_tg = compare_concepts(df, "target", "prediction", "target_vs_generated")
#out_ig, sum_ig = compare_concepts(df, "input", "prediction", "input_vs_generated")

summary_df = pd.DataFrame([sum_tg])
summary_df

In [ ]:
# ---- Extra biomedical faithfulness diagnostics for target vs generated ----

# Re-extract sets so we can compute set-based diagnostics cleanly
input_sets  = extract_cui_sets(df["input"])
target_sets = extract_cui_sets(df["target"])
gen_sets    = extract_cui_sets(df["prediction"])

# Hallucinations: concepts generated but NOT in target (reference abstract)
halluc_counts = np.array([len(g - t) for t, g in zip(target_sets, gen_sets)])
# Omitted: concepts in target missing from generated
omit_counts = np.array([len(t - g) for t, g in zip(target_sets, gen_sets)])

# Coverage: how much of target concepts are present in generated
coverage = np.array([
    (len(t & g) / len(t)) if len(t) else np.nan
    for t, g in zip(target_sets, gen_sets)
])

# How many cases have no concepts at all
empty_input  = sum(len(s) == 0 for s in input_sets)
empty_target = sum(len(s) == 0 for s in target_sets)
empty_gen    = sum(len(s) == 0 for s in gen_sets)
empty_both_target_gen = sum((len(t) == 0 and len(g) == 0) for t, g in zip(target_sets, gen_sets))

print("Empty concept sets:")
print("  input empty:", empty_input)
print("  target empty:", empty_target)
print("  generated empty:", empty_gen)
print("  target & generated both empty:", empty_both_target_gen)

print("\nHallucination/Omission diagnostics (per example counts):")
print("  Avg hallucinated CUIs (gen - target):", float(halluc_counts.mean()))
print("  Median hallucinated CUIs:", float(np.median(halluc_counts)))

print("  Avg omitted CUIs (target - gen):", float(omit_counts.mean()))
print("  Median omitted CUIs:", float(np.median(omit_counts)))

print("\nCoverage diagnostics:")
print("  Mean target coverage (|T∩G|/|T|):", float(np.nanmean(coverage)))
print("  Median target coverage:", float(np.nanmedian(coverage)))

In [ ]:
import numpy as np
import pandas as pd
import spacy
import scispacy

#BIOBART_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biov2bart/test_with_bertscore.csv"

BIOT5_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/plots/title_text_gen/bertscore/biot5/test_with_bertscore.csv"

df = pd.read_csv(BIOT5_PATH)

# -----------------------------
# Load model + add UMLS linker (NEW spaCy API)
# -----------------------------
nlp = spacy.load("en_core_sci_scibert")

# Add the scispaCy UMLS linker via factory name
# max_entities_per_mention=1 keeps top candidate only
nlp.add_pipe(
    "scispacy_linker",
    config={
        "linker_name": "umls",
        "resolve_abbreviations": True,
        "max_entities_per_mention": 1,
    },
)

print("Pipes:", nlp.pipe_names)

# -----------------------------
# Helpers
# -----------------------------
def cuis_from_doc(doc) -> set:
    """Extract unique UMLS CUIs from a processed spaCy doc."""
    cuis = set()
    for ent in doc.ents:
        # scispacy_linker stores candidates here
        if hasattr(ent._, "kb_ents"):
            for cui, score in ent._.kb_ents:
                cuis.add(cui)
    return cuis

def prf1(ref_set: set, pred_set: set):
    """
    Concept-level PRF1 based on set overlap.
    ref_set = CUIs from the "reference" text
    pred_set = CUIs from the "predicted/compared" text
    """
    if len(ref_set) == 0 and len(pred_set) == 0:
        return 1.0, 1.0, 1.0
    if len(ref_set) == 0 or len(pred_set) == 0:
        return 0.0, 0.0, 0.0

    tp = len(ref_set & pred_set)
    p = tp / len(pred_set) if len(pred_set) else 0.0
    r = tp / len(ref_set) if len(ref_set) else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) else 0.0
    return p, r, f1

def extract_cui_sets(texts, batch_size=32):
    texts = pd.Series(texts).fillna("").astype(str).tolist()
    return [cuis_from_doc(doc) for doc in nlp.pipe(texts, batch_size=batch_size)]

def compare_concepts(df_in, col_ref, col_pred, name, batch_size=32):
    """
    Compare CUIs between df_in[col_ref] (reference) and df_in[col_pred] (compared).
    Returns per-example CUIs + PRF1 and macro/micro aggregates.
    """
    ref_sets = extract_cui_sets(df_in[col_ref], batch_size=batch_size)
    pred_sets = extract_cui_sets(df_in[col_pred], batch_size=batch_size)

    P, R, F1 = [], [], []
    TP = FP = FN = 0

    for rs, ps in zip(ref_sets, pred_sets):
        p, r, f = prf1(rs, ps)
        P.append(p); R.append(r); F1.append(f)

        TP += len(rs & ps)
        FP += len(ps - rs)
        FN += len(rs - ps)

    out = pd.DataFrame({
        f"{name}_ref_cuis":  [";".join(sorted(s)) for s in ref_sets],
        f"{name}_pred_cuis": [";".join(sorted(s)) for s in pred_sets],
        f"{name}_P": P,
        f"{name}_R": R,
        f"{name}_F1": F1,
    })

    macro_P = float(np.mean(P))
    macro_R = float(np.mean(R))
    macro_F1 = float(np.mean(F1))

    micro_P = TP / (TP + FP) if (TP + FP) else 0.0
    micro_R = TP / (TP + FN) if (TP + FN) else 0.0
    micro_F1 = (2 * micro_P * micro_R / (micro_P + micro_R)) if (micro_P + micro_R) else 0.0

    summary = {
        "comparison": name,
        "N": int(len(df_in)),
        "macro_P": macro_P,
        "macro_R": macro_R,
        "macro_F1": macro_F1,
        "micro_P": float(micro_P),
        "micro_R": float(micro_R),
        "micro_F1": float(micro_F1),
    }
    return out, summary

# -----------------------------
# RUN: 3 comparisons you asked for
# -----------------------------
#out_it, sum_it = compare_concepts(df, "input", "target", "input_vs_target")
out_tg, sum_tg = compare_concepts(df, "target", "prediction", "target_vs_generated")
#out_ig, sum_ig = compare_concepts(df, "input", "prediction", "input_vs_generated")

summary_df = pd.DataFrame([sum_tg])
summary_df

In [ ]:
# ---- Extra biomedical faithfulness diagnostics for target vs generated ----

# Re-extract sets so we can compute set-based diagnostics cleanly
target_sets = extract_cui_sets(df["target"])
gen_sets    = extract_cui_sets(df["prediction"])

# Hallucinations: concepts generated but NOT in target (reference abstract)
halluc_counts = np.array([len(g - t) for t, g in zip(target_sets, gen_sets)])
# Omitted: concepts in target missing from generated
omit_counts = np.array([len(t - g) for t, g in zip(target_sets, gen_sets)])

# Coverage: how much of target concepts are present in generated
coverage = np.array([
    (len(t & g) / len(t)) if len(t) else np.nan
    for t, g in zip(target_sets, gen_sets)
])

# How many cases have no concepts at all
empty_target = sum(len(s) == 0 for s in target_sets)
empty_gen    = sum(len(s) == 0 for s in gen_sets)
empty_both_target_gen = sum((len(t) == 0 and len(g) == 0) for t, g in zip(target_sets, gen_sets))

print("Empty concept sets:")
print("  target empty:", empty_target)
print("  generated empty:", empty_gen)
print("  target & generated both empty:", empty_both_target_gen)

print("\nHallucination/Omission diagnostics (per example counts):")
print("  Avg hallucinated CUIs (gen - target):", float(halluc_counts.mean()))
print("  Median hallucinated CUIs:", float(np.median(halluc_counts)))

print("  Avg omitted CUIs (target - gen):", float(omit_counts.mean()))
print("  Median omitted CUIs:", float(np.median(omit_counts)))

print("\nCoverage diagnostics:")
print("  Mean target coverage (|T∩G|/|T|):", float(np.nanmean(coverage)))
print("  Median target coverage:", float(np.nanmedian(coverage)))

In [ ]:
# training code above ...

# auto-disconnect when done
from IPython.display import display, Javascript
display(Javascript('google.colab.runtime.disconnect()'))


<IPython.core.display.Javascript object>